# Load GUNDAM engine state from a ROOT output

This notebook loads the engine configuration from a GUNDAM output ROOT file, optionally injects the post-fit parameter state, and evaluates the likelihood at the restored point.

In [1]:
nCpuThreads = 3
gundamLibPath = "/Users/nadrino/Documents/Work/Install/gundam/lib"
workDir = "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024"

# GUNDAM output file containing gundam/config/unfoldedJson_TNamed and, if requested,
# FitterEngine/postFit/Migrad/parameterStateAfterMinimize_TNamed.
outputRootPath = "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/outputs/fit/gundamFitter_configOA2024_With_v12ProdRun45_onlyFlux_Asimov.root"

# Optional. If set, this config is used instead of the config stored in outputRootPath.
# outputRootPath can still be used to load the post-fit state.
configPath = None

overrideList = []
dataType = "Asimov"  # "Asimov", "Toy", or "RealData"
loadPostFitState = True
seed = 12345

In [2]:
import sys
import numpy as np
from pathlib import Path

repoRoot = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
srcPath = repoRoot / "src"
if srcPath.exists() and str(srcPath) not in sys.path:
    sys.path.insert(0, str(srcPath))

from gundam_interface import GundamInterface, GundamLoader, GundamRuntime

In [3]:
np.random.seed(seed)

runtime = GundamRuntime(
    loader=GundamLoader(gundamLibPath=gundamLibPath),
    workDir=workDir,
    nCpuThreads=nCpuThreads,
    configPath=configPath,
    outputRootPath=outputRootPath,
    loadPostFitState=loadPostFitState,
    overrideList=overrideList,
    dataType=dataType,
    randomSeed=seed,
)

runtime.toDict(includeConfigJsonString=False)

{'nCpuThreads': 3,
 'workDir': '/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024',
 'dataType': 'Asimov',
 'loader': {'moduleName': 'GUNDAM',
  'gundamLibPath': '/Users/nadrino/Documents/Work/Install/gundam/lib'},
 'randomSeed': 12345,
 'outputRootPath': '/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/outputs/fit/gundamFitter_configOA2024_With_v12ProdRun45_onlyFlux_Asimov.root',
 'loadPostFitState': True,
 'overrideList': []}

In [4]:
gundam = GundamInterface(runtime)
gundam.configure()
gundam.initialize()

2026.07.10 15:18:59  INFO ConfigUtils: Extracting config file for fitter file: /Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/outputs/fit/gundamFitter_configOA2024_With_v12ProdRun45_onlyFlux_Asimov.root
2026.07.10 15:19:00  WARN ConfigUtils: /fitterEngineConfig: key "allParamVariations" has been renamed. Use "paramVariationsSigmas" instead.
2026.07.10 15:19:00  WARN ConfigUtils: /fitterEngineConfig: key "scanConfig" has been renamed. Use "parameterScannerConfig" instead.
2026.07.10 15:19:00 ALERT ConfigUtils: /fitterEngineConfig: field "propagatorConfig" should be moved to: fitterEngineConfig/likelihoodInterfaceConfig/propagatorConfig
2026.07.10 15:19:00  WARN ConfigUtils: /fitterEngineConfig/minimizerConfig: key "max_iter" has been renamed. Use "maxIterations" instead.
2026.07.10 15:19:00  WARN ConfigUtils: /fitterEngineConfig/minimizerConfig: key "max_fcn" has been renamed. Use "maxFcnCalls" instead.
2026.07.10 15:19:00 ALERT ConfigUtils: /fitterEngineConfig/pr

In [5]:
print(f"Initialized GUNDAM with {len(gundam.parameters)} active parameters")
print(f"Loaded post-fit state: {runtime.loadPostFitState}")

for index, parameter in enumerate(gundam.parameters):
    print(
        f"{index:3d}: {parameter.name} "
        f"prior={parameter.prior:.8g} "
        f"step={parameter.stepSize:.8g} "
        f"value={parameter.value:.8g}"
    )

Initialized GUNDAM with 100 active parameters
Loaded post-fit state: True
  0: Flux Systematics/#0 prior=1 step=0.05924376 value=1
  1: Flux Systematics/#1 prior=1 step=0.052534918 value=1
  2: Flux Systematics/#2 prior=1 step=0.052933735 value=1
  3: Flux Systematics/#3 prior=1 step=0.051454976 value=1
  4: Flux Systematics/#4 prior=1 step=0.073819516 value=1
  5: Flux Systematics/#5 prior=1 step=0.087584986 value=1
  6: Flux Systematics/#6 prior=1 step=0.069835737 value=1
  7: Flux Systematics/#7 prior=1 step=0.049799122 value=1
  8: Flux Systematics/#8 prior=1 step=0.050205095 value=1
  9: Flux Systematics/#9 prior=1 step=0.070165897 value=1
 10: Flux Systematics/#10 prior=1 step=0.11426228 value=1
 11: Flux Systematics/#11 prior=1 step=0.097235633 value=1
 12: Flux Systematics/#12 prior=1 step=0.065676759 value=1
 13: Flux Systematics/#13 prior=1 step=0.071617626 value=1
 14: Flux Systematics/#14 prior=1 step=0.088904925 value=1
 15: Flux Systematics/#15 prior=1 step=0.076413478 va

In [6]:
currentValues = gundam.getParameterValues()
currentLlh = gundam.evaluateLlh()

print(f"Current LLH: {currentLlh:.8g}")
print("Current parameters:")
for name, physicalValue in zip(gundam.parameterNames, currentValues):
    print(f"  - {name}: physical={physicalValue:.8g}")

Current LLH: 3.2518425e-13
Current parameters:
  - Flux Systematics/#0: physical=1
  - Flux Systematics/#1: physical=1
  - Flux Systematics/#2: physical=1
  - Flux Systematics/#3: physical=1
  - Flux Systematics/#4: physical=1
  - Flux Systematics/#5: physical=1
  - Flux Systematics/#6: physical=1
  - Flux Systematics/#7: physical=1
  - Flux Systematics/#8: physical=1
  - Flux Systematics/#9: physical=1
  - Flux Systematics/#10: physical=1
  - Flux Systematics/#11: physical=1
  - Flux Systematics/#12: physical=1
  - Flux Systematics/#13: physical=1
  - Flux Systematics/#14: physical=1
  - Flux Systematics/#15: physical=1
  - Flux Systematics/#16: physical=1
  - Flux Systematics/#17: physical=1
  - Flux Systematics/#18: physical=1
  - Flux Systematics/#19: physical=1
  - Flux Systematics/#20: physical=1
  - Flux Systematics/#21: physical=1
  - Flux Systematics/#22: physical=1
  - Flux Systematics/#23: physical=1
  - Flux Systematics/#24: physical=1
  - Flux Systematics/#25: physical=1
 